# DecodeLabs Project 3
## Customer Segmentation using Unsupervised Learning

This notebook performs customer segmentation using K-Means clustering, PCA, Elbow Method and Silhouette Score.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_excel(file_name)
df.head()

In [ ]:
# Basic Information
print(df.info())
print(df.describe(include='all'))
print(df.isnull().sum())

In [ ]:
# Data Cleaning
df = df.drop_duplicates()

for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        mode = df[col].mode()
        if not mode.empty:
            df[col] = df[col].fillna(mode.iloc[0])

df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

In [ ]:
# Features
features = ['Quantity','UnitPrice','ItemsInCart','TotalPrice',
            'PaymentMethod','OrderStatus','ReferralSource',
            'Month','DayOfWeek']

X = df[features]

numeric = ['Quantity','UnitPrice','ItemsInCart','TotalPrice','Month','DayOfWeek']
categorical = ['PaymentMethod','OrderStatus','ReferralSource']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])

X_processed = preprocessor.fit_transform(X)

In [ ]:
# PCA
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_processed.toarray() if hasattr(X_processed,'toarray') else X_processed)

print('Components:', pca.n_components_)
print('Explained Variance:', pca.explained_variance_ratio_.sum())

In [ ]:
# Elbow Method
inertia = []
K = range(2,11)

for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca)
    inertia.append(km.inertia_)

plt.figure(figsize=(6,4))
plt.plot(K, inertia, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

In [ ]:
# Silhouette Scores
scores = {}
for k in range(2,11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    scores[k] = silhouette_score(X_pca, labels)

best_k = max(scores, key=scores.get)
print(scores)
print('Best K:', best_k)

In [ ]:
# Final Model
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_pca)

plt.figure(figsize=(7,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=df['Cluster'])
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.title('Customer Segments')
plt.show()

In [ ]:
# Customer Personas
summary = df.groupby('Cluster')[['Quantity','UnitPrice','ItemsInCart','TotalPrice']].mean()
print(summary)

df.to_csv('Project3_Customer_Segments.csv', index=False)
summary.to_csv('Cluster_Summary.csv')
print('Files Saved Successfully')